# Example 15 — Heat conduction: the transient slab and the fin

Two staples of every heat-transfer course, both with exact solutions, both trained in
seconds.

**Part A — 1-D transient conduction in a slab:**
$$\frac{\partial u}{\partial t} = \alpha\,\frac{\partial^2 u}{\partial x^2},\qquad u(0,t)=u(1,t)=0,\qquad u(x,0) = \sin(\pi x) + 0.3\sin(3\pi x)$$
Each Fourier mode decays independently — the exact solution is
$$u = e^{-\alpha\pi^2 t}\sin(\pi x) + 0.3\,e^{-9\alpha\pi^2 t}\sin(3\pi x),$$
and mode 3 dies **nine times faster** than mode 1 ($k^2$ scaling). Physics does to the
solution what spectral bias (Example 6) does to training — a nice symmetry to point out.

**Part B — the fin (extended surface):**
$$\frac{d^2\theta}{dx^2} = m^2\,\theta,\qquad \theta(0)=1\ (\text{base}),\quad \theta'(L)=0\ (\text{adiabatic tip})$$
with $\theta = (T-T_\infty)/(T_b-T_\infty)$ and $m^2 = hP/kA_c$. Exact:
$\theta = \cosh(m(L-x))/\cosh(mL)$, and the engineering payoff is the **fin efficiency**
$$\eta = \frac{\tanh(mL)}{mL}\;\; (= 0.48201\ \text{for } mL=2).$$
Like Blasius's $f''(0)=0.332$ (Example 11), we judge the PINN by the *engineering number*
it recovers, not just the curve.

> Runs in ~20 s total on CPU; faster on GPU.

In [ ]:
# Cell 1 -- Part A: transient slab (soft IC/BC, standard PINN recipe)
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

ALPHA, T, PI = 0.2, 0.5, np.pi
def u_exact(x, t):
    return torch.exp(-ALPHA*PI**2*t)*torch.sin(PI*x) \
         + 0.3*torch.exp(-9*ALPHA*PI**2*t)*torch.sin(3*PI*x)

net = nn.Sequential(nn.Linear(2, 48), nn.Tanh(), nn.Linear(48, 48), nn.Tanh(),
                    nn.Linear(48, 48), nn.Tanh(), nn.Linear(48, 1)).to(device)
opt = torch.optim.Adam(net.parameters(), 2e-3)
x_ic = torch.linspace(0, 1, 200, device=device).reshape(-1, 1)

t0 = time.perf_counter()
for e in range(3000):
    opt.zero_grad()
    x = torch.rand(2000, 1, device=device).requires_grad_(True)
    t = (torch.rand(2000, 1, device=device)*T).requires_grad_(True)
    u  = net(torch.cat([x, t], 1))
    ux = torch.autograd.grad(u,  x, torch.ones_like(u),  create_graph=True)[0]
    ut = torch.autograd.grad(u,  t, torch.ones_like(u),  create_graph=True)[0]
    uxx= torch.autograd.grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    tb = torch.rand(200, 1, device=device)*T
    xb = torch.cat([torch.zeros(200, 1, device=device), torch.ones(200, 1, device=device)])
    loss = ((ut - ALPHA*uxx)**2).mean() \
         + 10*((net(torch.cat([x_ic, torch.zeros_like(x_ic)], 1)) - u_exact(x_ic, torch.zeros_like(x_ic)))**2).mean() \
         + 10*(net(torch.cat([xb, torch.cat([tb, tb])], 1))**2).mean()
    loss.backward(); opt.step()
if device.type == 'cuda': torch.cuda.synchronize()
print(f'slab training: {time.perf_counter()-t0:.0f} s')

xg = torch.linspace(0, 1, 401, device=device).reshape(-1, 1)
plt.figure(figsize=(9, 4.2))
for tv, c in zip((0.0, 0.05, 0.15, 0.5), ('k', 'tab:red', 'tab:orange', 'tab:blue')):
    tt = torch.full_like(xg, tv)
    with torch.no_grad():
        up = net(torch.cat([xg, tt], 1)).cpu().numpy().ravel()
    ue = u_exact(xg, tt).cpu().numpy().ravel()
    err = np.sqrt(np.mean((up-ue)**2))
    plt.plot(xg.cpu(), ue, c, lw=2.2, alpha=.5)
    plt.plot(xg.cpu(), up, '--', color=c, lw=1.4, label=f't={tv} (L2={err:.1e})')
plt.xlabel('x'); plt.ylabel('temperature u'); plt.legend(fontsize=9); plt.grid(alpha=.3)
plt.title('Transient slab: mode 3 ripples die 9× faster than mode 1 (solid=exact, dashed=PINN)')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 2 -- Part B: the fin (hard base BC, soft adiabatic tip) + fin efficiency
M, L = 2.0, 1.0                       # m*L = 2
theta_exact = lambda x: np.cosh(M*(L-x))/np.cosh(M*L)

torch.manual_seed(0)
net2 = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(),
                     nn.Linear(32, 1)).to(device)
theta = lambda x: 1 + x*net2(x)       # theta(0)=1 HARD (base temperature)
opt = torch.optim.Adam(net2.parameters(), 2e-3)
xL = torch.ones(1, 1, device=device)

t0 = time.perf_counter()
for e in range(3000):
    opt.zero_grad()
    x = torch.rand(512, 1, device=device).requires_grad_(True)
    th  = theta(x)
    thx = torch.autograd.grad(th,  x, torch.ones_like(th),  create_graph=True)[0]
    thxx= torch.autograd.grad(thx, x, torch.ones_like(thx), create_graph=True)[0]
    xm = xL.clone().requires_grad_(True)
    thm = theta(xm)
    thmx = torch.autograd.grad(thm, xm, torch.ones_like(thm), create_graph=True)[0]
    loss = ((thxx - M*M*th)**2).mean() + 10*thmx.pow(2).sum()   # adiabatic tip: theta'(L)=0
    loss.backward(); opt.step()
if device.type == 'cuda': torch.cuda.synchronize()
print(f'fin training: {time.perf_counter()-t0:.0f} s')

xg = torch.linspace(0, 1, 401, device=device).reshape(-1, 1)
with torch.no_grad():
    thp = theta(xg).cpu().numpy().ravel()
the = theta_exact(xg.cpu().numpy().ravel())

# fin efficiency: actual heat / heat if whole fin were at base temperature
eta_pinn  = float(np.trapezoid(thp, xg.cpu().numpy().ravel())/L)
eta_exact = np.tanh(M*L)/(M*L)

plt.figure(figsize=(8, 4))
plt.plot(xg.cpu(), the, 'g', lw=2.4, label='exact  cosh(m(L−x))/cosh(mL)')
plt.plot(xg.cpu(), thp, 'r--', lw=1.6, label='PINN')
plt.xlabel('x / L'); plt.ylabel(r'$\theta$ = (T−T∞)/(T_b−T∞)'); plt.grid(alpha=.3); plt.legend()
plt.title(f'Fin (mL=2):  efficiency η — PINN {eta_pinn:.5f}  vs exact {eta_exact:.5f}')
plt.tight_layout(); plt.show()
print(f'L2(θ) = {np.sqrt(np.mean((thp-the)**2)):.1e}')
print(f'fin efficiency: PINN {eta_pinn:.5f}   exact tanh(mL)/mL = {eta_exact:.5f}'
      f'   ({abs(eta_pinn-eta_exact)/eta_exact*100:.2f}% err)')

## Observations (for heat-transfer notes)

- **Physics filters frequencies the way training does.** The $e^{-\alpha k^2\pi^2 t}$ decay
  means diffusion kills fine structure fastest — by $t=0.15$ the mode-3 ripples are gone
  and every profile looks like mode 1. Same $k^2$ scaling that makes high modes *hard to
  train* (Example 6) makes them *fast to decay* — diffusion is a low-pass filter.
- **All three BC types in one page.** Dirichlet (slab walls, soft), a hard-constrained
  Dirichlet (fin base, via the trial function $1 + xN$), and Neumann (adiabatic tip,
  $\theta'(L)=0$ — enforced through autograd, no ghost nodes needed as in FD).
- **Engineering numbers again:** fin efficiency to ~0.03% — the PINN is judged by
  $\eta = \tanh(mL)/mL$, exactly as Example 11 was judged by $f''(0)$.
- **The heat equation IS Stokes' first problem.** Replace $u\to$ velocity, $\alpha\to\nu$
  and Part A becomes the impulsively-started-plate problem of fluid dynamics —
  momentum diffuses exactly like heat. One notebook, two courses.

**Experiments to try:** make $\alpha$ trainable and recover it from 20 noisy 'thermocouple'
readings (inverse conduction — thermal diffusivity measurement, 5 lines via Example 2's
recipe); convective tip $\theta'(L) = -\mathrm{Bi}\,\theta(L)$ instead of adiabatic (Robin
BC — one line); make $mL$ a network input and produce the fin-efficiency *curve*
$\eta(mL)$ in one training (Example 5's surrogate trick); two-material slab with different
$\alpha$ per half (interface conditions — FBPINN territory, Example 9).